In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import os
from natsort import natsorted

import scanpy as sc
import seaborn as sns

from scroutines import basicu

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tools.sm_exceptions import ValueWarning
from tqdm import tqdm


import sys
sys.path.insert(0, '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/myvisctx/analysis_multiome/')
import lmm

In [4]:
outfigdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/'
f = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/superdupermegaRNA_hasraw_cheng22_astro_v2.h5ad'
adata = sc.read(f)
adata

AnnData object with n_obs × n_vars = 7697 × 15573
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'Time', 'Light', 'curated_cluster', 'pc1', 'pc2'
    var: 'feature_types'
    uns: 'leiden', 'ngbr_astro'
    obsm: 'pc_astro'
    layers: 'norm'
    obsp: 'ngbr_astro_connectivities', 'ngbr_astro_distances'

In [13]:
%%time

tag = 'd260129'
output = os.path.join(outfigdir, f'NRDR_DEGs_LMM_astro_cheng22_{tag}.h5ad')

exp_conds = ['P28', 'P28_dr', 'P38', 'P38_dr']
obs_fixed1 = 'Time'
obs_fixed2 = 'Light'
obs_random = 'Sample'


offset = 1e-2
scale = 1e4


# adatasub = adata[adata.obs['Age'].isin(exp_conds)]
adatasub = adata
# ### test
# adatasub = adatasub[:,:20]
# ### test

genes = adatasub.var.index.values 

obs = adatasub.obs[[obs_fixed1, obs_fixed2, obs_random]].copy()
obs = obs.dropna()

adatasub = adatasub[obs.index]

# mat (CP10k norm)
mat = np.array(adatasub.X.todense())/adatasub.obs['n_counts'].values.reshape(-1,1)*scale

res = lmm.run_lmm_two_fixed(mat, genes, obs, obs_fixed1, obs_fixed2, obs_random, output=output, offset=offset)

(7697, 15573) (7697, 3)
(7697, 15573) (7697, 3)
(7697, 11147) (7697, 3)


100% 11147/11147 [21:17<00:00,  8.73it/s]


CPU times: user 21min 17s, sys: 10.9 s, total: 21min 28s
Wall time: 21min 27s


In [14]:
print(output)

/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_astro_cheng22_d260129.h5ad


In [15]:
sc.read(output)

AnnData object with n_obs × n_vars = 11147 × 5
    obs: 'sigma', 'converge'
    layers: 'effsize', 'param', 'pval', 'qval'

In [16]:
res

AnnData object with n_obs × n_vars = 11147 × 5
    obs: 'sigma', 'converge'
    layers: 'pval', 'param', 'effsize', 'qval'

In [17]:
res.layers['pval']

array([[3.72584973e-02, 1.46559333e-01, 8.13634533e-02, 3.56371431e-01,
        2.85637396e-02],
       [3.19864106e-01, 3.62772078e-01, 9.09145108e-01, 4.22327715e-01,
        9.99995431e-01],
       [3.81072526e-01, 8.70786209e-01, 1.02828019e-01, 2.28418184e-01,
        1.76719482e-01],
       ...,
       [8.99469421e-03, 1.32801442e-02, 7.89998594e-03, 1.11267524e-01,
        3.92661659e-01],
       [6.00339164e-01, 5.85981322e-01, 6.89373543e-01, 3.07461477e-01,
        4.04202602e-02],
       [5.31565074e-03, 8.25500777e-01, 1.41714801e-04, 8.62253083e-01,
        3.98147207e-01]])

In [19]:
res.var

""
Intercept
Time[T.P38]
Light[T.NR]
Time[T.P38]:Light[T.NR]
Sample Var


In [20]:

# per subtype